In [1]:
import pandas as pd
from pathlib import Path
import pyarrow as pa  # not yet needed, might need it later
import pyarrow.parquet as pq
import pyarrow.compute as pc  # not yet needed, might need it later.

In [2]:
metrics_dir = Path('../../data/raw/parquet-metrics/')
# Note: We generally give a directory, rather than an individual file, to the next step.
metrics_pq = pq.ParquetDataset(metrics_dir)
metrics_df = metrics_pq.read().to_pandas()

In [4]:
systems_cleaned = pd.read_csv('../../data/core/systems_cleaned.csv')
all_data_systems = systems_cleaned[
    systems_cleaned['has_current_data']
    & systems_cleaned['has_irradiance_data']
    & systems_cleaned['has_power_data']
    & systems_cleaned['has_temperature_data']
    & systems_cleaned['has_voltage_data']
]
all_data_ids = set(all_data_systems.system_id)

parquet_systems = systems_cleaned.loc[
    systems_cleaned.loc[:, 'is_lake_parquet_data']
]  # is already boolean!
all_parquet_system_ids = list(parquet_systems.system_id.unique())
all_parquet_system_ids.sort()

all_rich_parquet_data_ids = set(all_data_systems.system_id.unique()).intersection(
    set(all_parquet_system_ids)
)
all_rich_parquet_data_ids = list(all_rich_parquet_data_ids)
all_rich_parquet_data_ids.sort()

In [6]:
j = 0
system_id = all_rich_parquet_data_ids[j]
relevant_rows_metrics = metrics_df[metrics_df['system_id'] == system_id]
relevant_rows_metrics

,system_id,metric_id,sensor_name,common_name,raw_units,units,calc_scale,calc_offset,calc_details,aggregation_type,source_type,source_id,comments,standard_name
1297,2,346,dc_power,DC power,W,W,1.0,0.0,,avg,NaN,NaN,,dc_power__346
1298,2,348,dc_pos_current,DC current,A,A,1.0,0.0,,avg,NaN,NaN,,dc_pos_current__348
1299,2,351,das_battery_voltage,DC voltage battery,V,V,1.0,0.0,,avg,NaN,NaN,,das_battery_voltage__351
1300,2,345,poa_irradiance,Irradiance POA,W/m^2,W/m^2,1.0,0.0,,avg,NaN,NaN,,poa_irradiance__345
1301,2,349,module_temp_1,Temperature module,C,C,1.0,0.0,,avg,NaN,NaN,,module_temp_1__349
1302,2,347,dc_pos_voltage,DC voltage,V,V,1.0,0.0,,avg,NaN,NaN,,dc_pos_voltage__347
1303,2,350,das_temp,Temperature panel,C,C,1.0,0.0,,avg,NaN,NaN,,das_temp__350


In [19]:
#for 'filtered' mode, add 'measured_on' to sensor_names you want.
# for test, intentionally loading in a weird order to experiment.
col_names = ['das_temp', 'dc_power', 'dc_pos_current']
col_names.insert(0, 'measured_on')
# for old access, access by metric_id.
my_metric_ids = [350, 346, 348]

In [20]:
source = 'filtered'  # 'raw', 'filtered', 'samples', or 'input'
if source == 'filtered':
    access_system_dir = Path(f'../../../data_ds_project/filtered/systems/parquet/{system_id}/')
    current_pq = pq.ParquetDataset(access_system_dir)
    current_df = current_pq.read(columns=col_names).to_pandas()
elif source == 'raw':  # raw data download from all_parquet_downloader
    access_system_dir = Path(f'../../../data_ds_project/systems/parquet/{system_id}/')
    current_pq = pq.ParquetDataset(access_system_dir,
                               filters= [
                                   ('metric_id', 'in', my_metric_ids) #in particular, only look a the my_metric_id's
                               ])
    current_df = current_pq.read().to_pandas()
elif source == 'samples':  # sample data in GitHub
    access_system_dir = Path(f'../../data/sorted_by_metric/systems/parquet/{system_id}/')
    current_pq = pq.ParquetDataset(access_system_dir,
                               filters= [
                                   ('metric_id', 'in', my_metric_ids) #in particular, only look a the my_metric_id's
                               ])
    current_df = current_pq.read().to_pandas()
else: 
    raise ValueError('Not a valid input!')

In [21]:
current_df

,measured_on,das_temp,dc_power,dc_pos_current
0,2010-01-21 11:02:00,5.564000,13.31000,0.054000
1,2010-01-21 11:03:00,5.573000,13.06000,0.053000
2,2010-01-21 11:04:00,5.591000,13.07000,0.053000
3,2010-01-21 11:05:00,5.610000,13.06000,0.053000
4,2010-01-21 11:06:00,5.625000,12.48000,0.051000
...,...,...,...,...
1910165,2020-01-13 16:25:00,9.044684,28.74832,0.156170
1910166,2020-01-13 16:30:00,8.955369,26.35972,0.142881
1910167,2020-01-13 16:35:00,8.866957,21.83201,0.118108
1910168,2020-01-13 16:40:00,8.782425,10.90820,0.055269


Ok, so the order of loading in the argument allows you to sort columns as you please.